# Notebook 8 — Ablation: state effects in the national RSF

Two matched experiments on the Notebook 5 design matrix:

1. **Do the 51 `STATE_*` dummies help or hurt?** (arms A/B) Geography already enters
   through the continuous climate and coastal features, so state identity may be
   redundant — or may even hurt within-state ranking by letting trees split on the
   state code instead of physical covariates.
2. **Does a dedicated single-state model beat the pooled model within its state?**
   (arm C) Pooled vs per-state C-index score different tasks — the pooled 0.8040 earns
   credit for between-state comparisons; a per-state C-index only scores within-state
   ranking, which is harder. MA's 0.6889 is 3rd-lowest of 50 but inside the 0.64–0.94
   national spread. Arm C asks how much of that gap a specialist trained only on MA
   recovers, holding features and test rows fixed.

**Arms** (A/B share one stratified 300k training subsample and the full ~241k test set;
C trains and evaluates on the MA subset of the same split):

| Arm | Training rows | Features | Question it answers |
|---|---|---|---|
| A | 300k national | current set (STATE_* kept) | baseline at matched cost |
| B | 300k national | STATE_* dropped | does removing state code help within-state ranking? |
| C | MA only (~16k) | national set, no STATE_* | does an MA specialist beat the pooled model on MA? |

C uses the national matrix's features, so it lacks the state-DOT maintenance features a
purpose-built MA model could add (`../ma_bridges/`).

**Decision rule:** drop STATE_* from the full model (`DROP_STATE_DUMMIES = True` in
`train_national_rsf.ipynb`) only if B beats A on median per-state C-index without losing
pooled C-index beyond noise (±0.005).

In [ ]:
# ── Configuration + typed load (same load pattern as train_national_rsf.ipynb) ──
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

DATA_CLEAN = Path("us_rsf_data_clean.csv")
OUT_JSON   = Path("us_ablation_state_dummies.json")
KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]

ABLATION_SAMPLE = 300_000
# Matched-cost arms: 100 trees x 60k rows/tree is ~1/3 the full Notebook 5 workload, enough
# for stable C-index comparisons. low_memory=True: only rsf.predict risk scores are needed
# here (no survival curves), and it roughly halves fit memory.
ABLATION_PARAMS = dict(n_estimators=100, max_samples=0.2, min_samples_split=100,
                       min_samples_leaf=50, max_features="sqrt", low_memory=True,
                       n_jobs=-1, random_state=42)
# MA-standalone capacity (../ma_bridges/RSF.ipynb: 1000 trees, leaf=5, split=10, full
# bootstrap on ~16k bridges); 300 trees give the same C-index at a third of the cost.
MA_PARAMS = dict(n_estimators=300, min_samples_split=10, min_samples_leaf=5,
                 max_features="sqrt", low_memory=True, n_jobs=-1, random_state=42)

cols = pd.read_csv(DATA_CLEAN, nrows=0).columns
dtypes = {c: "float32" for c in cols}
dtypes.update({k: str for k in KEYS})
dtypes["event"] = "int8"
df = pd.read_csv(DATA_CLEAN, dtype=dtypes)
for k in KEYS:
    df[k] = df[k].str.strip()

y   = Surv.from_arrays(event=df["event"].astype(bool).to_numpy(),
                       time=df["time"].astype("float64").to_numpy())
ids = df[KEYS].copy()
X = df.drop(columns=["event", "time", "bridge_age"] + KEYS)   # bridge_age == time; never a feature
del df

STATE_COLS = [c for c in X.columns if c.startswith("STATE_")]
assert STATE_COLS, "no STATE_* dummies in the matrix — nothing to ablate"
print(f"X: {X.shape[0]:,} x {X.shape[1]}   STATE_* dummies: {len(STATE_COLS)}")

In [ ]:
# ── Shared split + stratified ablation subsample ──────────────────────────────
# Same 80/20 split as Notebook 5 (random_state=42) -> every arm shares one test set.
X_train, X_test, y_train, y_test, ids_train, ids_test = \
    train_test_split(X, y, ids, test_size=0.2, random_state=42)

# Stratified (state, event) subsample keeps each state's event mix intact at 300k rows.
strat = pd.DataFrame({"state": ids_train["STATE_FIPS"].to_numpy(), "event": y_train["event"]})
pos = (strat.groupby(["state", "event"], group_keys=False)
            .sample(frac=min(1.0, ABLATION_SAMPLE / len(strat)), random_state=42)).index.to_numpy()
X_sub, y_sub = X_train.iloc[pos], y_train[pos]
print(f"ablation train: {len(X_sub):,}   shared test: {len(X_test):,}   "
      f"test event rate: {y_test['event'].mean()*100:.1f}%")

In [ ]:
# ── Arms A (with STATE_*) and B (without) ─────────────────────────────────────
def cindex_by_state(risk):
    """Per-state C-index with Notebook 5's stability filter (n>=200, events>=30)."""
    rows, state_arr = [], ids_test["STATE_FIPS"].to_numpy()
    for st in np.unique(state_arr):
        m = state_arr == st
        n, n_ev = int(m.sum()), int(y_test["event"][m].sum())
        if n < 200 or n_ev < 30:
            continue
        ci = concordance_index_censored(y_test["event"][m], y_test["time"][m], risk[m])[0]
        rows.append({"STATE_FIPS": st, "n_test": n, "n_events": n_ev, "c_index": ci})
    return pd.DataFrame(rows)

def run_arm(name, Xtr, ytr, Xte, params):
    t0 = time.time()
    rsf = RandomSurvivalForest(**params)
    rsf.fit(Xtr, ytr)
    risk = np.concatenate([rsf.predict(Xte.iloc[i:i+50_000])
                           for i in range(0, len(Xte), 50_000)])
    print(f"[{name}] {Xtr.shape[1]} features, fit+predict {(time.time()-t0)/60:.1f} min", flush=True)
    return risk

risk_A = run_arm("A: with STATE_*",    X_sub, y_sub, X_test, ABLATION_PARAMS)
risk_B = run_arm("B: without STATE_*", X_sub.drop(columns=STATE_COLS), y_sub,
                 X_test.drop(columns=STATE_COLS), ABLATION_PARAMS)

pooled, per_state = {}, {}
for name, risk in [("A", risk_A), ("B", risk_B)]:
    pooled[name] = concordance_index_censored(y_test["event"], y_test["time"], risk)[0]
    per_state[name] = cindex_by_state(risk)
    print(f"arm {name}: pooled C-index {pooled[name]:.4f}")

cmp = per_state["A"].merge(per_state["B"], on=["STATE_FIPS", "n_test", "n_events"],
                           suffixes=("_A", "_B"))
cmp["delta_B_minus_A"] = cmp["c_index_B"] - cmp["c_index_A"]
ma_row = cmp[cmp["STATE_FIPS"] == "25"]
print(f"\nper-state median: A {cmp['c_index_A'].median():.4f}   B {cmp['c_index_B'].median():.4f}")
print(f"median delta (B-A): {cmp['delta_B_minus_A'].median():+.4f}   "
      f"states better under B: {int((cmp['delta_B_minus_A'] > 0).sum())}/{len(cmp)}")
if len(ma_row):
    print(f"MA: A {ma_row['c_index_A'].iloc[0]:.4f} -> B {ma_row['c_index_B'].iloc[0]:.4f}")
cmp.sort_values("delta_B_minus_A")

In [ ]:
# ── Arm C: MA-only specialist benchmark ───────────────────────────────────────────────
# Same split membership, so C evaluates on exactly the MA rows of the shared test set —
# directly comparable to the national model's within-MA C-index (0.6889, Notebook 5).
ma_tr = ids_train["STATE_FIPS"].to_numpy() == "25"
ma_te = ids_test["STATE_FIPS"].to_numpy() == "25"
Xtr_ma = X_train[ma_tr].drop(columns=STATE_COLS)     # constant within one state
Xte_ma = X_test[ma_te].drop(columns=STATE_COLS)
ytr_ma, yte_ma = y_train[ma_tr], y_test[ma_te]
print(f"MA train: {len(Xtr_ma):,}   MA test: {len(Xte_ma):,}")

t0 = time.time()
rsf_C = RandomSurvivalForest(**MA_PARAMS).fit(Xtr_ma, ytr_ma)
ci_C = concordance_index_censored(yte_ma["event"], yte_ma["time"], rsf_C.predict(Xte_ma))[0]
print(f"C (MA-only specialist): {ci_C:.4f}   ({(time.time()-t0)/60:.1f} min)", flush=True)

In [ ]:
# ── Summary -> us_ablation_state_dummies.json ─────────────────────────────────
results = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "design": {
        "ablation_train_rows": int(len(X_sub)),
        "shared_test_rows": int(len(X_test)),
        "ablation_params": {k: (v if isinstance(v, (int, float, str, bool, type(None)))
                                else str(v)) for k, v in ABLATION_PARAMS.items()},
        "ma_params": {k: (v if isinstance(v, (int, float, str, bool, type(None)))
                          else str(v)) for k, v in MA_PARAMS.items()},
        "n_state_dummies": len(STATE_COLS),
    },
    "arm_A_with_state": {
        "pooled_c_index": round(float(pooled["A"]), 4),
        "median_state_c_index": round(float(cmp["c_index_A"].median()), 4),
        "ma_c_index": round(float(ma_row["c_index_A"].iloc[0]), 4) if len(ma_row) else None,
    },
    "arm_B_without_state": {
        "pooled_c_index": round(float(pooled["B"]), 4),
        "median_state_c_index": round(float(cmp["c_index_B"].median()), 4),
        "ma_c_index": round(float(ma_row["c_index_B"].iloc[0]), 4) if len(ma_row) else None,
    },
    "delta_summary": {
        "median_state_delta_B_minus_A": round(float(cmp["delta_B_minus_A"].median()), 4),
        "states_better_under_B": int((cmp["delta_B_minus_A"] > 0).sum()),
        "n_states_compared": int(len(cmp)),
    },
    "arm_C_ma_specialist_c_index": round(float(ci_C), 4),
}
OUT_JSON.write_text(json.dumps(results, indent=2))
print(f"saved {OUT_JSON}\n")
print(json.dumps(results, indent=2))